# Start here

**An abstaining hybrid agent for anemia types from CBC data.** Research prototype, not a diagnostic tool.

The project combines a cited rule engine (an expert system) with machine-learning models on a public complete blood count dataset. It is built in six phases. Phases 1-3 are done, and each has a notebook that **runs that phase** and then explains it:

| Notebook | What it runs | Time with default settings |
|---|---|---|
| `phase1_walkthrough.ipynb` | dataset check, cleaning, data report | under a minute |
| `phase2_walkthrough.ipynb` | rule engine and rules-only baseline | under a minute |
| `phase3_walkthrough.ipynb` | evaluation of all models, both calibration versions, diagnostic | a few minutes |

Every phase notebook first notes the committed results, re-runs the phase and then checks that the re-run reproduces them. The held-out test rows stay locked until the final evaluation (no notebook or script reads them).

This notebook checks your setup and runs the unit tests. Run its cells from top to bottom.

In [1]:
import os
import re
import subprocess
import sys
import time
from pathlib import Path

ROOT = Path.cwd().parent
sys.path.insert(0, str(ROOT / "src"))

import numpy as np
import pandas as pd
from IPython.display import Image, display

pd.set_option("display.width", 140)
pd.set_option("display.max_columns", 20)


def run_step(script, *args, show=25):
    """Run one project script from src/ with this notebook's Python, from the repository root."""
    t0 = time.perf_counter()
    env = {**os.environ, "PYTHONIOENCODING": "utf-8"}
    r = subprocess.run([sys.executable, "-X", "faulthandler=0", str(ROOT / "src" / script), *args], cwd=ROOT,
                       capture_output=True, text=True, encoding="utf-8", errors="replace", env=env)
    lines = (r.stdout or "").rstrip().splitlines()
    if show:
        if len(lines) > show:
            print(f"... ({len(lines) - show} earlier lines not shown)")
        print("\n".join(lines[-show:]))
    print(f"[{' '.join([script, *args])} finished in {time.perf_counter() - t0:.1f} s, exit code {r.returncode}]")
    if r.returncode != 0:
        print(r.stderr[-3000:])
        raise RuntimeError(f"{script} failed")


print("Research prototype, not a diagnostic tool. The test set is not used in this notebook.")

Research prototype, not a diagnostic tool. The test set is not used in this notebook.


## 1. Python and package versions

The versions this project was tested with are pinned in `requirements.txt`. A different version usually still works, but exact numbers can then differ in the last digit.

In [2]:
import importlib.metadata as im

print("Python", sys.version.split()[0], "(tested with 3.12.6)")
rows = []
for line in (ROOT / "requirements.txt").read_text(encoding="utf-8").splitlines():
    if "==" in line and not line.startswith("#"):
        name, want = line.strip().split("==")
        try:
            have = im.version(name)
        except im.PackageNotFoundError:
            have = "missing"
        rows.append((name, want, have, "ok" if have == want else "different"))
pd.DataFrame(rows, columns=["package", "tested", "installed", "status"])

Python 3.12.6 (tested with 3.12.6)


,package,tested,installed,status
0,numpy,2.1.3,2.1.3,ok
1,pandas,3.0.5,3.0.5,ok
2,scikit-learn,1.9.0,1.9.0,ok
3,scipy,1.18.0,1.18.0,ok
4,matplotlib,3.11.1,3.11.1,ok
5,pytest,9.2.0,9.2.0,ok
6,markdown,3.11.3,3.11.3,ok
7,imbalanced-learn,0.14.2,0.14.2,ok
8,joblib,1.5.3,1.5.3,ok
9,nbformat,5.11.1,5.11.1,ok


## 2. Is the dataset present and unchanged?

The CSV in `data/` is the original Kaggle file (see `data/NOTICE.md`). Its SHA-256 fingerprint must match the one recorded on 7 Nov 2026.

In [3]:
run_step("check_dataset.py")

file: diagnosed_cbc_data_v4.csv, bytes: 125590 (expected 125590)
sha256: f7dee8d4fe500aec6e8e4bfbeb6df6b83bc86bef03d540721e0aeaad2759fcdb
expected: f7dee8d4fe500aec6e8e4bfbeb6df6b83bc86bef03d540721e0aeaad2759fcdb
OK: dataset matches
[check_dataset.py finished in 0.4 s, exit code 0]


## 3. Unit tests

In [4]:
r = subprocess.run([sys.executable, "-m", "pytest", "-p", "no:faulthandler", "-q", "--color=no", "tests"], cwd=ROOT,
                   capture_output=True, text=True, encoding="utf-8", errors="replace")
print(r.stdout.strip().splitlines()[-1])
assert r.returncode == 0, r.stdout[-3000:]

78 passed in 1.46s


## 4. Optional: run all phase notebooks from here

Set `RUN_ALL_PHASE_NOTEBOOKS = True` to execute the three phase notebooks in order and save their outputs. `RUN_FULL_CV = True` also retrains every Phase 3 model from scratch, which takes much longer (see the README). Otherwise open the phase notebooks one by one.

In [5]:
RUN_ALL_PHASE_NOTEBOOKS = False
RUN_FULL_CV = False

if RUN_ALL_PHASE_NOTEBOOKS:
    env = {**os.environ, "ANEMIA_RUN_FULL_CV": "1" if RUN_FULL_CV else "0", "PYTHONIOENCODING": "utf-8"}
    for nb in ["phase1_walkthrough.ipynb", "phase2_walkthrough.ipynb", "phase3_walkthrough.ipynb"]:
        t0 = time.perf_counter()
        r = subprocess.run([sys.executable, "-m", "nbconvert", "--to", "notebook", "--execute", "--inplace",
                            "--ExecutePreprocessor.timeout=-1", nb], cwd=ROOT / "notebooks",
                           capture_output=True, text=True, encoding="utf-8", errors="replace", env=env)
        print(f"{nb}: exit code {r.returncode}, {time.perf_counter() - t0:.0f} s")
        if r.returncode != 0:
            print(r.stderr[-3000:])
            break
else:
    print("Skipped. Open the phase notebooks one by one, or set RUN_ALL_PHASE_NOTEBOOKS = True.")

Skipped. Open the phase notebooks one by one, or set RUN_ALL_PHASE_NOTEBOOKS = True.
